# Tutorial: Pricing Optimization with MLServe.com + Synthetic Demand Oracle

In this tutorial you will build an end-to-end **pricing system** in a controlled simulated environment:

1. **Generate data** from a hidden demand model (the `PricingOracle`).
2. **Estimate demand** using logistic regression:
   $$
   \Pr(\text{purchase}=1 \mid x, p)
   $$
3. **Deploy** your trained model to **MLServe.com** as a prediction service.
4. **Optimize prices** by scanning a discrete price grid and maximizing expected revenue:
   $$
   \mathbb{E}[R(p)\mid x] = p \cdot \widehat{\Pr}(\text{purchase}=1 \mid x,p)
   $$
5. **Evaluate** your pricing decisions using the oracle (which knows the true demand model).
6. **Send feedback** back to MLServe.com so that online metrics can be computed and compared across participants.

The objective is to maximize total revenue across a held-out test population.

---

## Cell 1 — Imports and environment setup

This cell loads the Python packages used throughout the notebook:

- **numpy/pandas** for data manipulation,
- **scikit-learn** for model training (logistic regression),
- **MLServeClient** for deployment and inference,
- **PricingOracle** for synthetic data generation and hidden evaluation,
- `.env` loading for credentials,
- `tqdm` for a progress bar during batched inference.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

from mlserve_sdk.client import MLServeClient
from oracle import PricingOracle

import os
from dotenv import load_dotenv

from tqdm import tqdm

load_dotenv()

True

---

## Cell 2 — Generate synthetic train/test data from the oracle

We instantiate the oracle and generate:

- a **training dataset** with features, logged prices, and purchase labels,
- a **test dataset** with features only (no labels, no prices).

We also retrieve the allowed **price grid** $\mathcal{P}$ that we will scan later for revenue optimization.

This keeps the exercise consistent for everyone: the oracle defines the market and the evaluation.

In [2]:
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

pricing_oracle = PricingOracle()
df_train = pricing_oracle.get_train(n=50000, seed=0)
df_test = pricing_oracle.get_test(n=5000, seed=1)

PRICE_GRID = pricing_oracle.allowed_price_grid()

---

## Cell 3 — Train a baseline demand model (logistic regression)

We prepare the data for modeling:

- `purchase` is the target label $y \in \{0,1\}$,
- `user_id` is an identifier and must be removed from the feature matrix,
- all remaining columns (including `price`) are model inputs.

We then fit a baseline logistic regression model and report validation accuracy as a quick sanity check.

> Note: accuracy is not the main objective here; we care about **revenue**. But accuracy is still useful for debugging.

In [3]:
TARGET_COL = "purchase"
DROP_COLS = ["user_id"]

X = df_train.drop(columns=[TARGET_COL])
y = df_train[TARGET_COL].astype(int)

for c in DROP_COLS:
    if c in X.columns:
        X = X.drop(columns=[c])

FEATURES = X.columns.tolist()
print("Features:", FEATURES)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, random_state=RANDOM_SEED)

model = LogisticRegression(max_iter=200)
model.fit(X_train, y_train)

accuracy = model.score(X_val, y_val)
print("Accuracy:", round(accuracy, 4))

Features: ['income', 'urgency', 'days_to_departure', 'is_business', 'loyalty', 'prev_purchases', 'price']
Accuracy: 0.8509


---

## Cell 4 — Authenticate to MLServe.com and define deployment metadata

We load credentials from environment variables and log in to MLServe.com.

We then choose:
- a `MODEL_NAME` shared across versions,
- a `MODEL_VERSION` (each student should pick a unique version name).

This is how we keep multiple student submissions organized under one model name.

In [5]:
USERNAME = os.getenv("USERNAME")
TOKEN = os.getenv("TOKEN")

client = MLServeClient()
client.login(USERNAME, TOKEN)

MODEL_NAME = "purchase_prob"
MODEL_VERSION = "Elon"

---

## Cell 5 — Deploy the model to MLServe.com and wait for readiness

We deploy the trained model to MLServe.com with:

- the model object,
- feature schema (column names),
- a background dataset (for monitoring / reference distribution),
- a basic metric dictionary (for display).

We then wait until deployment is ready before calling `predict()`.

In [6]:
resp = client.deploy(
    model=model,
    name=MODEL_NAME,
    version=MODEL_VERSION,
    features=FEATURES,
    background_df=X_train.sample(200, random_state=RANDOM_SEED),
    metrics={"accuracy": float(accuracy)},
    task_type="classification"
)

deployment_id = resp["deployment_id"]
print("🚀 Deployment started:", deployment_id)

final_status = client.wait_for_deployment(deployment_id)
print("✅ Final deployment status:", final_status)

🚀 Deployment started: 6
⏳ Deployment 6 status: building...
✅ Final deployment status: {'deployment_id': 6, 'model_name': 'purchase_prob', 'version': 'Elon', 'status': 'success', 'logs': 'Building Docker image...\n✅ Image built successfully. Starting container...\n🚀 Container deployed successfully.', 'created_at': '2026-01-17T22:06:11.710150+00:00', 'predict_url': 'https://mlserve.com/api/v1/predict/purchase_prob/Elon', 'feedback_url': 'https://mlserve.com/api/v1/feedback/purchase_prob/Elon'}


---

## Cell 6 — Build the user × price candidate set

To do pricing optimization we need predictions for each user under each candidate price.

We create a **cartesian product**:

- each test user is repeated for every price in the grid,
- each repeated row is augmented with a `price` column.

This yields a dataset of size:
$$
N_{\text{users}} \times |\mathcal{P}|
$$
which we will send to the deployed model for inference.

In [7]:
USER_ID_COL = "user_id"

df_users = df_test.copy()
# Build cartesian product: each user repeated for each price
rows = []
for p in PRICE_GRID:
    tmp = df_users.copy()
    tmp["price"] = p
    rows.append(tmp)

df_candidates = pd.concat(rows, ignore_index=True)
print(df_candidates.shape)
df_candidates.head()

(85000, 8)


,user_id,income,urgency,days_to_departure,is_business,loyalty,prev_purchases,price
0,4559655,0.505369,0.710110,0.101314,0,0,1,80.0
1,6595019,0.542744,0.765974,0.212360,0,1,0,80.0
2,9282616,0.392638,0.741968,0.528503,0,0,0,80.0
3,3068163,0.241077,0.021357,0.882064,0,0,1,80.0
4,1129730,0.610062,0.157447,0.846030,1,0,2,80.0


---

## Cell 7 — Predict in batches (MLServe.com limit: 500 rows per request)

The MLServe.com prediction endpoint accepts up to **500 records per request**.

We therefore:
- split `df_candidates` into chunks of 500 rows,
- call `client.predict()` on each chunk,
- accumulate `prediction_ids` and predicted probabilities in order.

It’s crucial that:
- batch results are appended in the same order as the candidate rows,
so that `(user_id, price)` aligns correctly with its prediction and id.

In [8]:
batch_size=500
pred_ids = []
p_hat = []

n = len(df_candidates)
for start in tqdm(range(0, n, batch_size)):
    end = min(start + batch_size, n)

    batch_df = df_candidates.iloc[start:end]

    TEST_DATA = {
        "features": FEATURES,
        "inputs": batch_df[FEATURES].values.tolist(),
    }

    batch_preds = client.predict(MODEL_NAME, MODEL_VERSION, TEST_DATA)

    pred_ids.extend(batch_preds["prediction_ids"])
    p_hat.extend(batch_preds["predictions"])

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████| 170/170 [00:28<00:00,  5.98it/s]


---

## Cell 8 — Compute expected revenue and select the best price per user

For each candidate row we compute:

- $ \hat p = \widehat{\Pr}(\text{purchase}=1\mid x,p)$
- expected revenue:
  $$
  \widehat{\mathbb{E}}[R] = p \cdot \hat p
  $$

Then for each user we select the price that maximizes expected revenue:

$$
p_i^* = \arg\max_{p \in \mathcal{P}} p \cdot \widehat{\Pr}(\text{purchase}=1\mid x_i,p)
$$

This produces `df_best`, containing exactly one chosen price per user.

In [9]:
df_pred = df_candidates[[USER_ID_COL, "price"]].copy()
df_pred["p_hat"] = np.array(p_hat, dtype=float)
df_pred["prediction_id"] = pred_ids
df_pred["expected_revenue"] = df_pred["price"] * df_pred["p_hat"]

# Pick best row per user
df_best = (
    df_pred.sort_values("expected_revenue", ascending=False)
           .groupby(USER_ID_COL, as_index=False)
           .head(1)
           .sort_values(USER_ID_COL)
           .reset_index(drop=True)
)
df_best

,user_id,price,p_hat,prediction_id,expected_revenue
0,1002431,260.0,1.0,cbe4d0b5-92a8-4acd-a923-a93b540dbcd0,260.0
1,1002717,320.0,0.0,f32e2edb-48ec-4a78-8dd3-2fb87a9c9a92,0.0
2,1005624,300.0,0.0,c3a53bb7-84ec-4e79-8788-bd5bdfc1c116,0.0
3,1008033,320.0,0.0,797d462c-a5a6-45a6-9434-264b1854cca9,0.0
4,1015092,300.0,0.0,c723639b-8c7d-4dc8-9e84-799860d96b94,0.0
...,...,...,...,...,...
4995,9994652,140.0,1.0,fea4dc55-2d85-4f72-831a-899392bb8794,140.0
4996,9994832,180.0,1.0,c8128c1e-f50c-4e7a-a6bf-c90b5e27f8c7,180.0
4997,9995146,160.0,1.0,0850e276-f493-47f7-9b4f-3a44c4535c75,160.0
4998,9995555,220.0,1.0,d357c6b4-3617-4e37-be4f-836e984a5d88,220.0


---

## Cell 9 — Evaluate chosen prices using the hidden oracle

Now we ask the oracle to score our chosen prices.

The oracle:
- uses the true (hidden) demand function,
- returns a binary purchase outcome and realized revenue per user.

This simulates the real-world situation where we can **choose prices**, observe outcomes, and measure business performance.

In [10]:
df_prices = df_best[[USER_ID_COL, "price"]].copy()
df_outcomes = pricing_oracle.score_prices(users=df_users, chosen_prices=df_prices)
df_outcomes

,user_id,purchase,revenue
0,4559655,0,0.0
1,6595019,0,0.0
2,9282616,0,0.0
3,3068163,0,0.0
4,1129730,0,0.0
...,...,...,...
4995,1163275,0,0.0
4996,1875920,0,0.0
4997,8624354,0,0.0
4998,9724039,0,0.0


---

## Cell 10 — Report business performance metrics

We compute simple summary statistics:

- total revenue (main ranking metric),
- average revenue per user,
- conversion rate.

These are the primary outputs you should track in pricing problems.

In [11]:
total_revenue = df_outcomes["revenue"].sum()
avg_revenue = df_outcomes["revenue"].mean()
conversion = df_outcomes["purchase"].mean()

print("TOTAL REVENUE:", round(total_revenue, 2))
print("AVG REVENUE/USER:", round(avg_revenue, 4))
print("CONVERSION RATE:", round(conversion, 4))

TOTAL REVENUE: 280820.0
AVG REVENUE/USER: 56.164
CONVERSION RATE: 0.2186


---

## Cell 11 — Send feedback back to MLServe.com

We now join:
- the oracle outcomes (purchase, revenue),
with
- the corresponding `prediction_id` for the chosen `(user, price)` record.

We then send feedback to MLServe.com:

- `true_value` = purchase outcome (0/1),
- `reward` = realized revenue.

This enables MLServe.com to compute and display **online metrics**, and allows the instructor to compare student submissions consistently.

In [14]:
# Join outcomes back to prediction ids for the selected rows
df_feedback = df_best[[USER_ID_COL, "prediction_id"]].merge(
    df_outcomes[[USER_ID_COL, "purchase", "revenue"]],
    on=USER_ID_COL,
    how="inner"
)

feedback_payload = [
    {
        "prediction_id": row["prediction_id"],
        "true_value": int(row["purchase"]),
        "reward": float(row["revenue"]),
    }
    for _, row in df_feedback.iterrows()
]

client.send_feedback(feedback_payload)
print("Sent feedback rows:", len(feedback_payload))

Sent feedback rows: 5000


---
## Cell 12 — Retrieve online metrics for all submitted versions

Finally, we fetch the list of all versions deployed under the shared model name.

For each version we retrieve online metrics (24h window) and concatenate them into a single table.

This table can serve as the instructor’s leaderboard input, typically using:
- mean reward,
- reward distribution,
- conversion rate,
- accuracy (secondary).

In [15]:
version_names=[v['version'] for v in client.list_model_versions(MODEL_NAME)]

metrics_dfs = []
for vn in version_names:
    metrics = client.get_online_metrics(
        MODEL_NAME,
        vn,
        window_hours=24,
        as_dataframe=True
    )
    metrics_dfs.append(metrics)

metrics_dfs = pd.concat(metrics_dfs)
metrics_dfs

,model,version,window_hours,n,n_supervised,accuracy,f1,brier,mean_reward,n_rewards
0,purchase_prob,Elon,24,85000,5000,0.792,0.671094,0.208,56.164,5000.0
0,purchase_prob,Tim,24,85000,0,NaN,NaN,NaN,NaN,NaN
0,purchase_prob,George,24,170000,5000,0.792,0.671094,0.208,56.164,5000.0
0,purchase_prob,Nick,24,170000,5000,0.792,0.671094,0.208,56.164,5000.0
